# IMDb movies dataset exploration

First analysis of the csv that is used as a reference for the movie-API.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

RAW_PATH = Path("../IMDb_movies.csv")

movie_data = pd.read_csv(RAW_PATH, low_memory=False)
print('Amount of movies:', movie_data.shape[0])

Amount of movies: 85855


In [2]:
movie_data.head()

,imdb_title_id,title,original_title,year,date_published,genre,duration,country,language,director,writer,production_company,actors,description,avg_vote,votes,budget,usa_gross_income,worlwide_gross_income,metascore,reviews_from_users,reviews_from_critics
0,tt0000009,Miss Jerry,Miss Jerry,1894,1894-10-09,Romance,45,USA,NaN,Alexander Black,Alexander Black,Alexander Black Photoplays,"Blanche Bayliss, William Courtenay, Chauncey D...",The adventures of a female reporter in the 1890s.,5.9,154,NaN,NaN,NaN,NaN,1.0,2.0
1,tt0000574,The Story of the Kelly Gang,The Story of the Kelly Gang,1906,1906-12-26,"Biography, Crime, Drama",70,Australia,NaN,Charles Tait,Charles Tait,J. and N. Tait,"Elizabeth Tait, John Tait, Norman Campbell, Be...",True story of notorious Australian outlaw Ned ...,6.1,589,$ 2250,NaN,NaN,NaN,7.0,7.0
2,tt0001892,Den sorte drøm,Den sorte drøm,1911,1911-08-19,Drama,53,"Germany, Denmark",NaN,Urban Gad,"Urban Gad, Gebhard Schätzler-Perasini",Fotorama,"Asta Nielsen, Valdemar Psilander, Gunnar Helse...",Two men of high rank are both wooing the beaut...,5.8,188,NaN,NaN,NaN,NaN,5.0,2.0
3,tt0002101,Cleopatra,Cleopatra,1912,1912-11-13,"Drama, History",100,USA,English,Charles L. Gaskill,Victorien Sardou,Helen Gardner Picture Players,"Helen Gardner, Pearl Sindelar, Miss Fielding, ...",The fabled queen of Egypt's affair with Roman ...,5.2,446,$ 45000,NaN,NaN,NaN,25.0,3.0
4,tt0002130,L'Inferno,L'Inferno,1911,1911-03-06,"Adventure, Drama, Fantasy",68,Italy,Italian,"Francesco Bertolini, Adolfo Padovan",Dante Alighieri,Milano Film,"Salvatore Papa, Arturo Pirovano, Giuseppe de L...",Loosely adapted from Dante's Divine Comedy and...,7.0,2237,NaN,NaN,NaN,NaN,31.0,14.0


## Summary statistics

In [3]:
movie_data.describe()

,duration,avg_vote,votes,metascore,reviews_from_users,reviews_from_critics
count,85855.000000,85855.000000,8.585500e+04,13305.000000,78258.000000,74058.000000
mean,100.351418,5.898656,9.493490e+03,55.896881,46.040826,27.479989
std,22.553848,1.234987,5.357436e+04,17.784874,178.511411,58.339158
min,41.000000,1.000000,9.900000e+01,1.000000,1.000000,1.000000
25%,88.000000,5.200000,2.050000e+02,43.000000,4.000000,3.000000
50%,96.000000,6.100000,4.840000e+02,57.000000,9.000000,8.000000
75%,108.000000,6.800000,1.766500e+03,69.000000,27.000000,23.000000
max,808.000000,9.900000,2.278845e+06,100.000000,10472.000000,999.000000


In [4]:
movie_data.describe(include="object")

/tmp/ipykernel_55729/626032987.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  movie_data.describe(include="object")


,imdb_title_id,title,original_title,year,date_published,genre,country,language,director,writer,production_company,actors,description,budget,usa_gross_income,worlwide_gross_income
count,85855,85855,85855,85855,85855,85855,85791,84954,85768,84283,81400,85786,83740,23710,15326,31016
unique,85855,82094,80852,113,22012,1257,4907,4376,34733,66859,32050,85729,83611,4642,14857,30414
top,tt0000009,Anna,Anna,2017,2010,Drama,USA,English,Jesús Franco,Jing Wong,Metro-Goldwyn-Mayer (MGM),"Nobuyo Ôyama, Noriko Ohara, Michiko Nomura, Ka...",The story of,$ 1000000,$ 1000000,$ 8144
freq,1,10,10,3329,113,12543,28511,35939,87,84,1284,9,15,758,19,15


## Missing values

In [5]:
movie_data.isna().sum().sort_values(ascending=False)

metascore                72550
usa_gross_income         70529
budget                   62145
worlwide_gross_income    54839
reviews_from_critics     11797
reviews_from_users        7597
production_company        4455
description               2115
writer                    1572
language                   901
director                    87
actors                      69
country                     64
imdb_title_id                0
genre                        0
date_published               0
year                         0
original_title               0
title                        0
duration                     0
avg_vote                     0
votes                        0
dtype: int64

## Which columns are integers?

pandas reads integer columns that have missing values as `float64` (NaN is a float), so `dtype` alone is misleading.
Instead, each column is classified by its non-null values:

- **int**: every value is numeric and a whole number
- **float**: every value is numeric, some have decimals
- **mixed**: mostly numeric, but some values are not
- **text**: no numeric values

In [6]:
def classify(column):
    values = column.dropna()
    numeric = pd.to_numeric(values, errors="coerce")
    non_numeric = values[numeric.isna()]

    if numeric.notna().sum() == 0:
        kind = "text"
    elif len(non_numeric) > 0:
        kind = "mixed"
    elif (numeric % 1 == 0).all():
        kind = "int"
    else:
        kind = "float"

    return pd.Series({
        "pandas_dtype": str(column.dtype),
        "kind": kind,
        "nulls": column.isna().sum(),
        "non_numeric": len(non_numeric) if kind == "mixed" else 0,
        "non_numeric_examples": list(non_numeric.unique()[:3]) if kind == "mixed" else [],
    })


types = movie_data.apply(classify).T
types

,pandas_dtype,kind,nulls,non_numeric,non_numeric_examples
imdb_title_id,str,text,0,0,[]
title,str,mixed,0,85760,"[Miss Jerry, The Story of the Kelly Gang, Den ..."
original_title,str,mixed,0,85750,"[Miss Jerry, The Story of the Kelly Gang, Den ..."
year,str,mixed,0,1,[TV Movie 2019]
date_published,str,mixed,0,81293,"[1894-10-09, 1906-12-26, 1911-08-19]"
genre,str,text,0,0,[]
duration,int64,int,0,0,[]
country,str,text,64,0,[]
language,str,text,901,0,[]
director,str,text,87,0,[]


## Cases to clean

In [7]:
movie_data.loc[pd.to_numeric(movie_data["year"], errors="coerce").isna(), ["imdb_title_id", "title", "year", "date_published"]]

,imdb_title_id,title,year,date_published
83917,tt8206668,Bad Education,TV Movie 2019,TV Movie 2019


### Clean and save

1. `year`: one row has `"TV Movie 2019"`; the year is extracted from the text.
2. Integer columns stored as decimals (e.g. `metascore = "99.0"`) are converted to plain integers.
   `INT_COLUMNS` must match `MovieRepository.INT_FIELDS` in the backend.

The CSV is re-read with every column as text, so saving it only changes the cells fixed here
(a default read would save `"None"` as an empty cell and `70` as `70.0`).

The result is written next to the original with a `_clean` suffix (`IMDb_movies_clean.csv`), so the original dataset stays untouched.

In [ ]:
movies_as_text = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)

malformed_year = ~movies_as_text["year"].str.fullmatch(r"\d{4}")
print("Before:")
print(movies_as_text.loc[malformed_year, ["imdb_title_id", "title", "year"]], end="\n\n")

movies_as_text.loc[malformed_year, "year"] = movies_as_text.loc[malformed_year, "year"].str.extract(r"(\d{4})", expand=False)

print("After:")
print(movies_as_text.loc[malformed_year, ["imdb_title_id", "title", "year"]])

assert movies_as_text["year"].str.fullmatch(r"\d{4}").all(), "there are still non-numeric years"

In [9]:
INT_COLUMNS = ["year", "duration", "votes", "metascore"]

for col in INT_COLUMNS:
    whole_number_decimals = movies_as_text[col].str.fullmatch(r"\d+\.0")
    print(f"{col}: {whole_number_decimals.sum()} values like '99.0' converted to '99'")
    movies_as_text.loc[whole_number_decimals, col] = movies_as_text.loc[whole_number_decimals, col].str.removesuffix(".0")

    int_or_empty = movies_as_text[col].str.fullmatch(r"\d*")
    assert int_or_empty.all(), f"{col} has non-integer values: {movies_as_text.loc[~int_or_empty, col].unique()[:5]}"

year: 0 values like '99.0' converted to '99'
duration: 0 values like '99.0' converted to '99'
votes: 0 values like '99.0' converted to '99'
metascore: 13305 values like '99.0' converted to '99'


In [10]:
OUTPUT_PATH = RAW_PATH.with_stem(f"{RAW_PATH.stem}_clean")

movies_as_text.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(movies_as_text)} movies to {OUTPUT_PATH}")

Saved 85855 movies to ../IMDb_movies_clean.csv


## Money columns

`budget` and the gross income columns are text: an amount with a currency prefix (e.g. `$ 2250`).

In [11]:
money_columns = ["budget", "usa_gross_income", "worlwide_gross_income"]

for col in money_columns:
    currency = movie_data[col].dropna().str.split().str[0]
    print(f"{col}: {currency.nunique()} currencies")
    print(currency.value_counts().head(5).to_string(), end="\n\n")

budget: 79 currencies
budget
$      16602
EUR     2422
INR      906
GBP      743
CAD      632

usa_gross_income: 4 currencies
usa_gross_income
$      15321
GBP        3
EUR        1
HKD        1

worlwide_gross_income: 5 currencies
worlwide_gross_income
$      30955
INR       58
GBP        1
NPR        1
PKR        1

